# Importing necessary libraries

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load dataset
df = pd.read_csv('nasdaq_ipos_merged.csv')

In [3]:
df.shape

(2845, 13)

In [4]:
df.isnull().sum()

dealID                            0
proposedTickerSymbol_x          950
companyName_x                   949
proposedExchange                949
proposedSharePrice              960
sharesOffered                   954
pricedDate                      949
dollarValueOfSharesOffered_x    950
dealStatus                      949
proposedTickerSymbol_y          619
companyName_y                   133
filedDate                       133
dollarValueOfSharesOffered_y    137
dtype: int64

In [5]:
# Merge the columns
df['companyName'] = df['companyName_x'].combine_first(df['companyName_y'])
df['proposedTickerSymbol'] = df['proposedTickerSymbol_x'].combine_first(df['proposedTickerSymbol_y'])
df['dollarValueOfSharesOffered'] = df['dollarValueOfSharesOffered_x'].combine_first(df['dollarValueOfSharesOffered_y'])

# Drop the original columns
df = df.drop(columns=[
    'companyName_x', 'companyName_y',
    'proposedTickerSymbol_x', 'proposedTickerSymbol_y',
    'dollarValueOfSharesOffered_x', 'dollarValueOfSharesOffered_y'
])


In [6]:
df.head()

,dealID,proposedExchange,proposedSharePrice,sharesOffered,pricedDate,dealStatus,filedDate,companyName,proposedTickerSymbol,dollarValueOfSharesOffered
0,1006008-95511,NaN,NaN,NaN,NaN,NaN,1/15/2021,LumiraDx Ltd,NaN,"$100,000,000"
1,1007622-101060,NaN,NaN,NaN,NaN,NaN,11/08/2021,5.11 ABR CORP.,NaN,"$100,000,000"
2,1008124-100682,NASDAQ Global,20.00,"9,075,000",10/29/2021,Priced,10/08/2021,"Entrada Therapeutics, Inc.",TRDA,"$181,500,000"
3,1010055-100948,NYSE,9.00,"289,150,555",12/09/2021,Priced,11/01/2021,Nu Holdings Ltd.,NU,"$2,602,354,995"
4,1010448-115077,NASDAQ Global,15.00,"19,000,000",9/11/2025,Priced,8/22/2025,LB PHARMACEUTICALS INC,LBRX,"$285,000,000"


In [7]:
df.isnull().sum()

dealID                          0
proposedExchange              949
proposedSharePrice            960
sharesOffered                 954
pricedDate                    949
dealStatus                    949
filedDate                     133
companyName                     0
proposedTickerSymbol          486
dollarValueOfSharesOffered      4
dtype: int64

In [8]:
df = df.dropna(subset=['filedDate', 'dollarValueOfSharesOffered' ])

# Replace nulls in 'dealStatus' with 'filed'
df['dealStatus'] = df['dealStatus'].fillna('filed')

# Replace nulls in 'pricedDate' with '03/31/2026'
df['pricedDate'] = df['pricedDate'].fillna('03/31/2026')

df['proposedExchange'] = df['proposedExchange'].fillna('Undecided')
df['proposedTickerSymbol'] = df['proposedTickerSymbol'].fillna('Not Decided')

df['proposedSharePrice'] = df['proposedSharePrice'].fillna('0.00')

df['sharesOffered'] = df['sharesOffered'].fillna('0')



In [9]:
df.isnull().sum()

dealID                        0
proposedExchange              0
proposedSharePrice            0
sharesOffered                 0
pricedDate                    0
dealStatus                    0
filedDate                     0
companyName                   0
proposedTickerSymbol          0
dollarValueOfSharesOffered    0
dtype: int64

In [10]:
df.shape

(2708, 10)

In [11]:
df.head()

,dealID,proposedExchange,proposedSharePrice,sharesOffered,pricedDate,dealStatus,filedDate,companyName,proposedTickerSymbol,dollarValueOfSharesOffered
0,1006008-95511,Undecided,0.00,0,03/31/2026,filed,1/15/2021,LumiraDx Ltd,Not Decided,"$100,000,000"
1,1007622-101060,Undecided,0.00,0,03/31/2026,filed,11/08/2021,5.11 ABR CORP.,Not Decided,"$100,000,000"
2,1008124-100682,NASDAQ Global,20.00,"9,075,000",10/29/2021,Priced,10/08/2021,"Entrada Therapeutics, Inc.",TRDA,"$181,500,000"
3,1010055-100948,NYSE,9.00,"289,150,555",12/09/2021,Priced,11/01/2021,Nu Holdings Ltd.,NU,"$2,602,354,995"
4,1010448-115077,NASDAQ Global,15.00,"19,000,000",9/11/2025,Priced,8/22/2025,LB PHARMACEUTICALS INC,LBRX,"$285,000,000"


In [12]:
# Clean and convert the columns
df['sharesOffered_M'] = (df['sharesOffered'].replace({',': ''}, regex=True) .astype(float, errors='ignore') / 1_000_000).round(2)
  # convert to millions)

df['dollarValueOfSharesOffered_M'] = (df['dollarValueOfSharesOffered'].replace({'\$': '', ',': ''}, regex=True).astype(float, errors='ignore') / 1_000_000).round(2)
   # convert to millions)

# Drop the original columns
df.drop(['sharesOffered', 'dollarValueOfSharesOffered'], axis=1, inplace=True)

# Check results
df.head()


<>:5: SyntaxWarning: invalid escape sequence '\$'
<>:5: SyntaxWarning: invalid escape sequence '\$'
C:\Users\paura\AppData\Local\Temp\ipykernel_5756\695061622.py:5: SyntaxWarning: invalid escape sequence '\$'
  df['dollarValueOfSharesOffered_M'] = (df['dollarValueOfSharesOffered'].replace({'\$': '', ',': ''}, regex=True).astype(float, errors='ignore') / 1_000_000).round(2)


,dealID,proposedExchange,proposedSharePrice,pricedDate,dealStatus,filedDate,companyName,proposedTickerSymbol,sharesOffered_M,dollarValueOfSharesOffered_M
0,1006008-95511,Undecided,0.00,03/31/2026,filed,1/15/2021,LumiraDx Ltd,Not Decided,0.00,100.00
1,1007622-101060,Undecided,0.00,03/31/2026,filed,11/08/2021,5.11 ABR CORP.,Not Decided,0.00,100.00
2,1008124-100682,NASDAQ Global,20.00,10/29/2021,Priced,10/08/2021,"Entrada Therapeutics, Inc.",TRDA,9.07,181.50
3,1010055-100948,NYSE,9.00,12/09/2021,Priced,11/01/2021,Nu Holdings Ltd.,NU,289.15,2602.35
4,1010448-115077,NASDAQ Global,15.00,9/11/2025,Priced,8/22/2025,LB PHARMACEUTICALS INC,LBRX,19.00,285.00


In [13]:
df.shape

(2708, 10)

In [14]:
df['filedDate'] = pd.to_datetime(df['filedDate'], errors='coerce')
df['pricedDate'] = pd.to_datetime(df['pricedDate'], errors='coerce')

In [15]:

# Numeric columns (in millions)
df['sharesOffered_M'] = pd.to_numeric(df['sharesOffered_M'], errors='coerce').round(2)
df['dollarValueOfSharesOffered_M'] = pd.to_numeric(df['dollarValueOfSharesOffered_M'], errors='coerce').round(2)
df['proposedSharePrice'] = pd.to_numeric(df['proposedSharePrice'], errors='coerce').round(2)
# Categorical / Text columns
text_cols = ['companyName', 'industry', 'exchange', 'status']
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype('category')

# Boolean columns (if any like withdrawn, amended etc.)
if 'withdrawn' in df.columns:
    df['withdrawn'] = df['withdrawn'].astype('boolean')

# Check results
print(df.dtypes)
df.head()


dealID                                  object
proposedExchange                        object
proposedSharePrice                     float64
pricedDate                      datetime64[ns]
dealStatus                              object
filedDate                       datetime64[ns]
companyName                           category
proposedTickerSymbol                    object
sharesOffered_M                        float64
dollarValueOfSharesOffered_M           float64
dtype: object


,dealID,proposedExchange,proposedSharePrice,pricedDate,dealStatus,filedDate,companyName,proposedTickerSymbol,sharesOffered_M,dollarValueOfSharesOffered_M
0,1006008-95511,Undecided,0.0,2026-03-31,filed,2021-01-15,LumiraDx Ltd,Not Decided,0.00,100.00
1,1007622-101060,Undecided,0.0,2026-03-31,filed,2021-11-08,5.11 ABR CORP.,Not Decided,0.00,100.00
2,1008124-100682,NASDAQ Global,20.0,2021-10-29,Priced,2021-10-08,"Entrada Therapeutics, Inc.",TRDA,9.07,181.50
3,1010055-100948,NYSE,9.0,2021-12-09,Priced,2021-11-01,Nu Holdings Ltd.,NU,289.15,2602.35
4,1010448-115077,NASDAQ Global,15.0,2025-09-11,Priced,2025-08-22,LB PHARMACEUTICALS INC,LBRX,19.00,285.00


In [17]:
df['companyName'] = df['companyName'].str.strip()
df['proposedExchange'] = df['proposedExchange'].str.strip()
df['dealStatus'] = df['dealStatus'].str.title().str.strip()

In [18]:
# Drop unwanted columns
df.drop(['dealID', 'proposedTickerSymbol'], axis=1, inplace=True, errors='ignore')

# Check to confirm
df.head()


,proposedExchange,proposedSharePrice,pricedDate,dealStatus,filedDate,companyName,sharesOffered_M,dollarValueOfSharesOffered_M
0,Undecided,0.0,2026-03-31,Filed,2021-01-15,LumiraDx Ltd,0.00,100.00
1,Undecided,0.0,2026-03-31,Filed,2021-11-08,5.11 ABR CORP.,0.00,100.00
2,NASDAQ Global,20.0,2021-10-29,Priced,2021-10-08,"Entrada Therapeutics, Inc.",9.07,181.50
3,NYSE,9.0,2021-12-09,Priced,2021-11-01,Nu Holdings Ltd.,289.15,2602.35
4,NASDAQ Global,15.0,2025-09-11,Priced,2025-08-22,LB PHARMACEUTICALS INC,19.00,285.00


In [19]:
df.rename(columns={
    'companyName': 'CompanyName',
    'proposedExchange': 'Exchange',
    'filedDate': 'FiledDate',
    'pricedDate': 'PricedDate',
    'dealStatus': 'DealStatus',
    'sharesOffered_M': 'SharesOffered(M)',
    'proposedSharePrice': 'SharePrice',
    'dollarValueOfSharesOffered_M': 'TotalValue(M)'
}, inplace=True)

df = df[['CompanyName', 'Exchange', 'FiledDate', 'PricedDate', 'DealStatus', 
         'SharesOffered(M)', 'SharePrice', 'TotalValue(M)']]

In [20]:
output_path = 'nasdaq_ipos_cleaned.csv'
df.to_csv(output_path, index=False)

In [1]:
import pandas as pd
import numpy as np

In [4]:
# Load dataset
df = pd.read_csv('C:\\Users\\paura\\ipo_project\\nasdaq_ipos_merged.csv')

In [5]:
df

,dealID,proposedTickerSymbol_x,companyName_x,proposedExchange,proposedSharePrice,sharesOffered,pricedDate,dollarValueOfSharesOffered_x,dealStatus,proposedTickerSymbol_y,companyName_y,filedDate,dollarValueOfSharesOffered_y
0,1006008-95511,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,LumiraDx Ltd,1/15/2021,"$100,000,000"
1,1007622-101060,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.11 ABR CORP.,11/08/2021,"$100,000,000"
2,1008124-100682,TRDA,"Entrada Therapeutics, Inc.",NASDAQ Global,20.00,"9,075,000",10/29/2021,"$181,500,000",Priced,TRDA,"Entrada Therapeutics, Inc.",10/08/2021,"$181,500,000"
3,1010055-100948,NU,Nu Holdings Ltd.,NYSE,9.00,"289,150,555",12/09/2021,"$2,602,354,995",Priced,NU,Nu Holdings Ltd.,11/01/2021,"$2,602,354,995"
4,1010448-115077,LBRX,LB PHARMACEUTICALS INC,NASDAQ Global,15.00,"19,000,000",9/11/2025,"$285,000,000",Priced,LBRX,LB PHARMACEUTICALS INC,8/22/2025,"$285,000,000"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2840,998057-95557,BVS,Bioventus Inc.,NASDAQ Global Select,13.00,"8,000,000",2/11/2021,"$104,000,000",Priced,BVS,Bioventus Inc.,1/20/2021,"$104,000,000"
2841,999152-103237,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"NYIAX, INC.",6/01/2022,"$10,637,500"
2842,999152-107203,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"NYIAX, INC.",7/27/2023,"$9,688,752"
2843,999173-100696,JUNS,"JUPITER NEUROSCIENCES, INC.",NASDAQ Capital,4.00,"2,750,000",12/03/2024,"$11,000,000",Priced,JUNS,"JUPITER NEUROSCIENCES, INC.",10/12/2021,"$11,000,000"


In [6]:
# Load dataset
df = pd.read_csv('C:\\Users\\paura\\ipo_project\\nasdaq_ipos_cleaned.csv')

In [7]:
df

,CompanyName,Exchange,FiledDate,PricedDate,DealStatus,SharesOffered(M),SharePrice,TotalValue(M)
0,LumiraDx Ltd,Undecided,2021-01-15,2026-03-31,Filed,0.00,0.0,100.00
1,5.11 ABR CORP.,Undecided,2021-11-08,2026-03-31,Filed,0.00,0.0,100.00
2,"Entrada Therapeutics, Inc.",NASDAQ Global,2021-10-08,2021-10-29,Priced,9.07,20.0,181.50
3,Nu Holdings Ltd.,NYSE,2021-11-01,2021-12-09,Priced,289.15,9.0,2602.35
4,LB PHARMACEUTICALS INC,NASDAQ Global,2025-08-22,2025-09-11,Priced,19.00,15.0,285.00
...,...,...,...,...,...,...,...,...
2703,Bioventus Inc.,NASDAQ Global Select,2021-01-20,2021-02-11,Priced,8.00,13.0,104.00
2704,"NYIAX, INC.",Undecided,2022-06-01,2026-03-31,Filed,0.00,0.0,10.64
2705,"NYIAX, INC.",Undecided,2023-07-27,2026-03-31,Filed,0.00,0.0,9.69
2706,"JUPITER NEUROSCIENCES, INC.",NASDAQ Capital,2021-10-12,2024-12-03,Priced,2.75,4.0,11.00
